In [1]:
import numpy as np
from PIL import Image


In [2]:
def replace_color(
    input_path: str,
    output_path: str,
    target_color: tuple[int, int, int] | tuple[int, int, int, int],
    replacement_color: tuple[int, int, int, int],
    tolerance: int = 0,
) -> None:
    """Replaces all instances of target_color with replacement_color in an image.

    :param input_path: Path to the source image.
    :param output_path: Destination path for the modified image (.png recommended).
    :param target_color: (R, G, B) or (R, G, B, A) color to replace (0-255).
    :param replacement_color: (R, G, B, A) replacement color (0-255).
    :param tolerance: Threshold variation per RGB channel (0 for exact match).
    """
    # Open image and ensure RGBA mode so transparency is supported
    img = Image.open(input_path).convert("RGBA")
    data = np.array(img)

    # Extract color channels
    r, g, b = data[..., 0], data[..., 1], data[..., 2]
    tr, tg, tb = target_color[:3]

    if tolerance == 0:
        match_mask = (r == tr) & (g == tg) & (b == tb)
    else:
        # Match colors within a tolerance window (useful for anti-aliased/JPEG edges)
        match_mask = (
            (np.abs(r.astype(int) - tr) <= tolerance)
            & (np.abs(g.astype(int) - tg) <= tolerance)
            & (np.abs(b.astype(int) - tb) <= tolerance)
        )

    # If target specifies an alpha channel, factor that into the match
    if len(target_color) == 4:
        match_mask = match_mask & (data[..., 3] == target_color[3])

    # Assign the new RGBA color to matching pixels
    data[match_mask] = replacement_color

    # Save output
    output_img = Image.fromarray(data, mode="RGBA")
    output_img.save(output_path, format="PNG")

In [7]:
def hex_to_rgba(hex_code: str, default_alpha: int = 255) -> tuple[int, int, int, int]:
    """Converts 6-digit or 8-digit hex strings into an RGBA tuple."""
    hex_code = hex_code.lstrip("#")
    if len(hex_code) == 6:
        r, g, b = (int(hex_code[i : i + 2], 16) for i in (0, 2, 4))
        return (r, g, b, default_alpha)
    elif len(hex_code) == 8:
        return tuple(int(hex_code[i : i + 2], 16) for i in (0, 2, 4, 6))
    raise ValueError(f"Invalid hex color format: {hex_code}")


def replace_hex_color(
    input_path: str,
    output_path: str,
    target_hex: str = "#000000",
    replacement_hex: str = "#1A76F9",
    tolerance: int = 0,
) -> None:
    """Replaces occurrences of target_hex with replacement_hex in an image and saves as PNG."""
    target_rgba = hex_to_rgba(target_hex)
    replacement_rgba = hex_to_rgba(replacement_hex)

    img = Image.open(input_path).convert("RGBA")
    data = np.array(img)

    r, g, b = data[..., 0], data[..., 1], data[..., 2]
    tr, tg, tb = target_rgba[:3]

    if tolerance == 0:
        match_mask = (r == tr) & (g == tg) & (b == tb)
    else:
        match_mask = (
            (np.abs(r.astype(int) - tr) <= tolerance)
            & (np.abs(g.astype(int) - tg) <= tolerance)
            & (np.abs(b.astype(int) - tb) <= tolerance)
        )

    # Respect source alpha if the target specifies an explicit alpha value
    if len(target_hex.lstrip("#")) == 8:
        match_mask = match_mask & (data[..., 3] == target_rgba[3])

    data[match_mask] = replacement_rgba

    output_img = Image.fromarray(data, mode="RGBA")
    output_img.save(output_path, format="PNG")

In [8]:
input_locations = ["/home/david/Downloads/scales.drawio.png", "/home/david/Downloads/checklist.drawio.png", "/home/david/Downloads/book.drawio.png", "/home/david/Downloads/spyglass.drawio.png", "/home/david/Downloads/clusters.drawio.png"]

output_locations = ["/home/david/Git/SDLT_Number_Crunching/Images/scales.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/checklist.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/book.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/spyglass.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/clusters.drawio.png"]

WHITE = (255, 255, 255)
TRANSPARENT = (0, 0, 0, 0)

for location in output_locations:

    # # Set tolerance=0 for exact white, or e.g. tolerance=10-20 to clean up near-white compression artifacts
    # replace_color(
    #     input_path=location,
    #     output_path=location,
    #     target_color=(0, 0, 0),
    #     replacement_color=(26, 118, 249),
    #     tolerance=5,
    # )


    replace_hex_color(
        input_path=location,
        output_path=location,
        target_hex="#FFFFFF",
        replacement_hex="#1A76F9",
        tolerance=15,  # Catches dark anti-aliased edges often found around black text/shapes
    )